# Mnemo: train the HOI-TCN pick-up / put-down model on public datasets

This notebook trains Mnemo's deep model **using only public datasets** (no personal photos or videos):

| Dataset | What it gives us | Access | Size |
|---|---|---|---|
| **Charades-Ego** | ~7,500 *take/put* segments, first-person, filmed **in homes**, 70 people | free, direct download | annotations 2.4 MB, videos (480p) 11.4 GB |
| **EPIC-KITCHENS-100** | precise *take/put* segments, first-person, kitchens | free (official download script) | very large; we download a few participants only |
| **Ego4D (FHO)** | *take/put* actions in many daily scenes | needs your approved licence (AWS keys from the email) | large; we download a subset of clips |

**What gets trained:** only HOI-TCN (pick-up vs put-down). Hands come from MediaPipe (already trained), object names from YOLO-World (no training) and places from CLIP (no training).

**Before you start:** Runtime → Change runtime type → **T4 GPU**.
Everything important (features, models) is saved to Google Drive, and extraction skips finished videos,
so if Colab disconnects, re-run the cells and it continues where it stopped.

## 1. Google Drive + project code
On your laptop, in the project folder, create the code zip:
```bash
python -m tools.make_colab_zip
```
Upload the resulting `mnemo_code.zip` to **My Drive/mnemo/** and run the next cell.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/mnemo'          # everything persistent lives here
FEAT = f'{WORK}/features'
MODELS = f'{WORK}/models'
SCRATCH = '/content/scratch'                    # big temporary downloads (local disk, deleted later)
for d in (WORK, FEAT, MODELS, SCRATCH):
    os.makedirs(d, exist_ok=True)

assert os.path.exists(f'{WORK}/mnemo_code.zip'), 'Upload mnemo_code.zip to My Drive/mnemo/ first'
!rm -rf /content/mnemo && mkdir -p /content/mnemo && cd /content/mnemo && unzip -q {WORK}/mnemo_code.zip
%cd /content/mnemo
!ls

## 2. Install + quick self-test (about 1-2 min)

In [ ]:
!pip install -q mediapipe ultralytics open_clip_torch pytest
import torch, mediapipe, ultralytics
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| mediapipe', mediapipe.__version__,
      '| ultralytics', ultralytics.__version__)
# The same unit tests as on the laptop: state machine, features, dataset parsers, causal model
!python -m pytest -q tests/test_interaction.py tests/test_features_metrics.py tests/test_epic_parsing.py tests/test_model.py::test_causal_and_streaming_equivalent

## 3. Settings
Measured on the laptop: hands ~23 ms + YOLO ~77 ms (CPU) per frame. On Colab YOLO runs on the GPU, so expect roughly 15-20 frames/s, but check the frames/s printed after each video.
Data needed per video: Charades ~30 s = 300 frames; EPIC 10 min = 6,000 frames.

In [ ]:
# First run: about 2.5-3 hours of extraction on a T4 (estimate; each video prints its real frames/s).
# Extraction skips finished videos, so later sessions can raise these numbers to add more data.
USE_CHARADES = True
CHARADES_MAX_VIDEOS = 300        # of ~2,500 first-person videos with take/put (~30 s each)

USE_EPIC = True
# If EPIC-KITCHENS *videos* (.MP4) are already in your Drive, put that folder here
# (sub-folders are searched). Leave '' to download them. Run the check cell below first.
EPIC_LOCAL_DIR = ''             # e.g. '/content/drive/MyDrive/EPIC-KITCHENS'
EPIC_PARTICIPANTS = ['P01', 'P02', 'P03']          # each participant = separate group for train/val/test
EPIC_MAX_VIDEOS_PER_PARTICIPANT = 3
EPIC_MAX_MINUTES = 10            # only the first N minutes of each (long) video

USE_EGO4D = False                # set True once your Ego4D licence/AWS keys arrive
EGO4D_MAX_CLIPS = 200


## 3b. Check the EPIC / VISOR folder you already have in Drive
Set `FOLDER` to the folder in your Drive and run. It tells you whether it contains **videos** (usable)
or only **VISOR sparse frames/masks** (not usable for this model, because it needs continuous video).

In [ ]:
FOLDER = '/content/drive/MyDrive/'   # ← change to your EPIC / VISOR folder
import collections, glob, os
counts = collections.Counter()
examples = {}
for p in glob.glob(os.path.join(FOLDER, '**', '*'), recursive=True):
    ext = os.path.splitext(p)[1].lower()
    if ext in ('.mp4', '.zip', '.jpg', '.png', '.json', '.csv', '.tar'):
        counts[ext] += 1
        examples.setdefault(ext, p)
print('files found:', dict(counts))
for ext, p in examples.items():
    print(f'  e.g. {ext}: {p}')
if counts['.mp4']:
    print(f"\n✅ {counts['.mp4']} videos found → set EPIC_LOCAL_DIR = '{FOLDER}' in the settings cell and re-run it.")
else:
    print('\n❌ No .MP4 videos. VISOR (sparse frames + masks) cannot train this model; '
          'leave EPIC_LOCAL_DIR empty so the EPIC cell downloads the videos.')

## 4. Charades-Ego (free, home scenes)
Downloads the 480p videos (11.4 GB) to the Colab disk, extracts features to Drive, then deletes the videos.

In [ ]:
if USE_CHARADES:
    CE = f'{SCRATCH}/charades'
    os.makedirs(CE, exist_ok=True)
    !wget -q -nc -O {CE}/CharadesEgo.zip https://ai2-public-datasets.s3-us-west-2.amazonaws.com/charades/CharadesEgo.zip
    !cd {CE} && unzip -q -o CharadesEgo.zip
    if not os.path.exists(f'{CE}/videos_done'):
        !wget -q --show-progress -nc -O {CE}/CharadesEgo_v1_480.tar https://ai2-public-datasets.s3-us-west-2.amazonaws.com/charades/CharadesEgo_v1_480.tar
        !cd {CE} && tar -xf CharadesEgo_v1_480.tar && rm CharadesEgo_v1_480.tar && touch videos_done
    !python -m model.extract_features charades --videos {CE} --annotations {CE}/CharadesEgo/CharadesEgo_v1_train.csv --classes {CE}/CharadesEgo/Charades_v1_classes.txt --max-videos {CHARADES_MAX_VIDEOS} --out {FEAT}

## 5. EPIC-KITCHENS-100 (free, kitchens, precise timings)
Downloads one participant at a time with the official script, extracts, deletes. The download server
can be slow; previously finished files are skipped if you re-run.

In [ ]:
if USE_EPIC:
    EK = f'{SCRATCH}/epic'
    os.makedirs(EK, exist_ok=True)
    base = 'https://raw.githubusercontent.com/epic-kitchens/epic-kitchens-100-annotations/master'
    !wget -q -nc -O {EK}/EPIC_100_train.csv {base}/EPIC_100_train.csv
    !wget -q -nc -O {EK}/EPIC_100_verb_classes.csv {base}/EPIC_100_verb_classes.csv
    if not os.path.exists(f'{EK}/epic-kitchens-download-scripts'):
        !git clone -q --depth 1 https://github.com/epic-kitchens/epic-kitchens-download-scripts.git {EK}/epic-kitchens-download-scripts

    import csv
    from collections import defaultdict
    vids = defaultdict(list)
    for r in csv.DictReader(open(f'{EK}/EPIC_100_train.csv')):
        if r['verb_class'] in ('0', '1') and r['video_id'] not in vids[r['participant_id']]:
            vids[r['participant_id']].append(r['video_id'])

    for p in EPIC_PARTICIPANTS:
        if EPIC_LOCAL_DIR:     # videos already in Drive: no download, no delete
            print(f'=== {p}: using videos in {EPIC_LOCAL_DIR}')
            !python -m model.extract_features epic --videos {EPIC_LOCAL_DIR} --annotations {EK}/EPIC_100_train.csv --verb-classes {EK}/EPIC_100_verb_classes.csv --participants {p} --max-videos {EPIC_MAX_VIDEOS_PER_PARTICIPANT} --max-minutes {EPIC_MAX_MINUTES} --out {FEAT}
            continue
        chosen = ','.join(sorted(vids[p])[:EPIC_MAX_VIDEOS_PER_PARTICIPANT])
        print(f'=== {p}: {chosen}')
        !cd {EK}/epic-kitchens-download-scripts && python epic_downloader.py --videos --specific-videos {chosen} --output-path {EK}/videos
        !python -m model.extract_features epic --videos {EK}/videos --annotations {EK}/EPIC_100_train.csv --verb-classes {EK}/EPIC_100_verb_classes.csv --participants {p} --max-minutes {EPIC_MAX_MINUTES} --out {FEAT}
        !rm -rf {EK}/videos


## 6. Ego4D FHO (after your licence is approved)
The approval email gives you an AWS access key + secret. They are typed in here (not saved in the notebook).
The annotation format is checked when loading; if Ego4D changed field names, the error message prints
the fields it found. Send that message to me.

In [ ]:
if USE_EGO4D:
    import getpass, glob, json, random
    !pip install -q ego4d awscli
    os.makedirs('/root/.aws', exist_ok=True)
    key_id = getpass.getpass('AWS access key id: ')
    secret = getpass.getpass('AWS secret access key: ')
    with open('/root/.aws/credentials', 'w') as f:
        f.write(f'[default]\naws_access_key_id = {key_id}\naws_secret_access_key = {secret}\n')

    E4 = f'{SCRATCH}/ego4d'
    !ego4d --output_directory={E4} --datasets annotations --benchmarks FHO -y
    ann = sorted(glob.glob(f'{E4}/**/fho_lta_train.json', recursive=True))
    print('annotation files:', ann)
    assert ann, 'fho_lta_train.json not found; run `!ls -R ' + E4 + '` and send me the listing'

    data = json.load(open(ann[0]))
    recs = data.get('clips', data) if isinstance(data, dict) else data
    clips = sorted({r['clip_uid'] for r in recs if str(r.get('verb', '')).lower().startswith(('take', 'put'))})
    random.Random(0).shuffle(clips)
    open(f'{E4}/clip_uids.txt', 'w').write('\n'.join(clips[:EGO4D_MAX_CLIPS]))
    print(f'{len(clips)} clips have take/put; downloading {min(len(clips), EGO4D_MAX_CLIPS)}')

    !ego4d --output_directory={E4} --datasets clips --benchmarks FHO --video_uid_file {E4}/clip_uids.txt -y
    !python -m model.extract_features ego4d --videos {E4} --annotations {ann[0]} --out {FEAT}

## 7. Check the extracted data

In [ ]:
import glob, numpy as np
for ds in sorted(os.listdir(FEAT)):
    files = glob.glob(f'{FEAT}/{ds}/*.npz')
    if not files:
        continue
    frames = ev = pick = put = 0
    groups = set()
    for f in files:
        z = np.load(f)
        frames += len(z['times']); ev += int((z['event'] > 0).sum())
        pick += int((z['gt_action'] == 'pickup').sum()); put += int((z['gt_action'] == 'putdown').sum())
        groups.add(os.path.basename(f).split('__')[0])
    print(f'{ds:10s} {len(files):5d} videos | {len(groups):4d} people/sources | {frames/36000:6.1f} h at 10 fps | '
          f'pickups {pick:5d} | putdowns {put:5d} | event frames {100*ev/max(frames,1):.1f}%')

## 8. Train HOI-TCN
Split is **by person / source video** inside each dataset (15% validation, 15% test), so the model is
always tested on people it has never seen. The best epoch and decision threshold are chosen on
validation only.

In [ ]:
dirs = ' '.join(f'{FEAT}/{d}' for d in sorted(os.listdir(FEAT)) if d != 'own' and glob.glob(f'{FEAT}/{d}/*.npz'))
print('training on:', dirs)
!python -m model.train --features {dirs} --out {MODELS}/hoi_tcn_public.pt --device cuda --epochs 40 --samples-per-epoch 8000 --batch-size 64 --patience 8

## 9. Test-set results (report these numbers)

In [ ]:
!python -m model.evaluate --checkpoint {MODELS}/hoi_tcn_public.pt --features {dirs} --split test --json {MODELS}/test_results_public.json

## 10. Training curve (for the report)

In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open(f'{MODELS}/hoi_tcn_public_history.json'))
ep = [x['epoch'] for x in h]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ep, [x['loss'] for x in h]); ax[0].set_title('training loss'); ax[0].set_xlabel('epoch')
for k in ('overall', 'pickup', 'putdown'):
    ax[1].plot(ep, [x[k]['f1'] for x in h], label=k)
ax[1].set_title('validation event F1 (+-1 s)'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.savefig(f'{MODELS}/training_curve.png', dpi=150); plt.show()

## 11. (Optional) fine-tune on your own chest-camera footage
Only if you later record and label some footage (`python -m model.extract_features own` on the laptop,
then copy `data/features/own` to `My Drive/mnemo/features/own`).

In [ ]:
if glob.glob(f'{FEAT}/own/*.npz'):
    !python -m model.train --features {FEAT}/own --init {MODELS}/hoi_tcn_public.pt --lr 3e-4 --out {MODELS}/hoi_tcn_own.pt --device cuda --epochs 30 --samples-per-epoch 3000
    !python -m model.evaluate --checkpoint {MODELS}/hoi_tcn_own.pt --features {FEAT}/own --split test
else:
    print('No own footage yet: skip.')

## 12. Use the model on the laptop
Download `My Drive/mnemo/models/hoi_tcn_public.pt` (or `hoi_tcn_own.pt`) to the project's `models/`
folder as **`models/hoi_tcn.pt`**, then:
```bash
python mnemo.py run some_video.mp4 --mode model --show
python mnemo.py ask "where is my phone?"
```